# Referral graph — local cleaning, first pass
Run locally; do not share the notebook after execution or patient records. The notebook does not print example rows, IDs, diagnoses, locations, or exact dates. Review even aggregate output before sharing; small populations may still be sensitive.

This pass preserves source rows, checks types and missingness, and proposes event groups. It does not train, impute, resolve conflicting diagnoses, or calculate history using the full dataset. Set the configuration below, then run in order. Dependencies: pandas, numpy, openpyxl; install them in your own environment if needed.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

DATA_PATH = Path("./data/Base_DataSet _version-09-30-2025_Bhaskar.xlsx")
SHEET_NAME = "Final_Base_Data_09_30_2025"  # Explicit source worksheet.
DATE_FORMAT = "%Y-%m-%d"  # Set the actual format; e.g. %m/%d/%Y. ISO timestamps are supported below.
DATE_IS_EXCEL_SERIAL = False  # Only True if numeric Excel dates were confirmed.
MISSING_TOKENS = {""}  # Add confirmed suppression/null tokens; do not guess that 0 or NA is missing.
OUTPUT_DIR = Path("local_cleaning_outputs")
WRITE_LOCAL_FILES = False  # Includes row-level private data; leave False for the initial pass.


## 1. Load as strings and normalize whitespace
This preserves textual IDs and ICD codes. It cannot restore leading zeros already lost by the source file. No automatic case folding or code correction is applied.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError("Set DATA_PATH to your local source file before running.")
if DATA_PATH.suffix.lower() in {".xlsx", ".xlsm"}:
    raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, dtype="string", keep_default_na=False)
elif DATA_PATH.suffix.lower() == ".csv":
    raw = pd.read_csv(DATA_PATH, dtype="string", keep_default_na=False)
else:
    raise ValueError("Use .xlsx, .xlsm, or .csv for this starter.")
names = raw.columns.astype(str).str.strip()
if names.duplicated().any():
    raise ValueError("Duplicate headers after trimming; resolve locally before continuing.")
raw.columns = names
df = raw.copy()
for col in df:
    df[col] = df[col].str.strip().mask(df[col].str.strip().isin(MISSING_TOKENS))
required = ["mso_membership_id", "primary_care_provider_npi", "rendering_provider_npi", "service_from_date", "DOC-DOC-LINK"]
missing_columns = [c for c in required if c not in df]
if missing_columns:
    raise ValueError("Missing required column names: " + ", ".join(missing_columns))
summary = {"source_rows": len(df), "source_columns": len(df.columns),
           "exact_duplicate_rows_beyond_first": int(df.duplicated().sum())}
print(json.dumps(summary, indent=2))


## 2. Coverage and numeric checks
Only counts are displayed. Invalid numeric values are reported and preserved in the string source, not silently replaced. Count fields may have suppression codes; confirm those before changing MISSING_TOKENS.

In [ ]:
KNOWN_ABSENT_COLUMNS = ["Tot_Srvcs_pcp", "Rndrng_Prvdr_Type_IMPR"]
SPECIALTY_PRIMARY_SOURCE = "clinician specialty"
SPECIALTY_COMPARISON_SOURCE = "rnd_pri_spec"  # Compare before reconciliation; no automatic fallback.
print("Previously confirmed absent columns:", KNOWN_ABSENT_COLUMNS)

numeric_cols = ["Age", "practice size_pcp", "years in medicare_pcp", "Tot_Benes_pcp",
 "practice size", "years in medicare", "Tot_Benes", "Tot_Srvcs", "Bene_Avg_Age", "Bene_Avg_Risk_Scre",
 "Bene_Age_LT_65_Cnt", "Bene_Age_65_74_Cnt", "Bene_Age_75_84_Cnt", "Bene_Age_GT_84_Cnt",
 "Bene_Feml_Cnt", "Bene_Male_Cnt", "final MIPS score", "Rating_RP", "Reviews_avg_rating_RP",
 "Number_of_ratings_RP", "D_pat_rnd_dist", "D_pcp_rnd_dist"]
categorical_cols = ["principal_diagnosis_code", "Gender", "IPA_CARRIER", "IPA_CARRIER_PRODUCT",
 "clinician specialty", "rnd_pri_spec", "Credential_RP",
 "Rndrng_Prvdr_Crdntls", "Credential_PCP", "Rndrng_Prvdr_Crdntls_pcp"]
coverage = pd.DataFrame([{"column": c, "present": c in df,
 "missing_count": int(df[c].isna().sum()) if c in df else None,
 "distinct_count": int(df[c].nunique()) if c in df else None}
 for c in numeric_cols + categorical_cols])
display(coverage)
numeric = {}
checks=[]
for c in numeric_cols:
    if c not in df: continue
    v = pd.to_numeric(df[c], errors="coerce")
    finite = v.notna() & np.isfinite(v.fillna(0).astype(float))
    numeric[c] = v.where(finite)
    checks.append({"column": c, "unparseable_count": int((df[c].notna() & v.isna()).sum()),
                   "nonfinite_count": int((v.notna() & ~finite).sum()),
                   "negative_count": int((v < 0).sum()),
                   "fractional_count": int((finite & v.mod(1).ne(0)).sum())})
display(pd.DataFrame(checks))


## 3. Labels, identifiers, and dates
Original regression negatives are retained in memory but excluded from positive-event construction. No target label is used as a feature. Dates are parsed explicitly; numeric dates require the configuration switch.

In [ ]:
label = pd.to_numeric(df["DOC-DOC-LINK"], errors="coerce")
source_date = df["service_from_date"]
if DATE_IS_EXCEL_SERIAL:
    dates = pd.to_datetime(pd.to_numeric(source_date, errors="coerce"), unit="D", origin="1899-12-30", errors="coerce")
else:
    dates = pd.to_datetime(source_date, format=DATE_FORMAT, errors="coerce")
    iso = source_date.str.match(r"^\d{4}-\d{2}-\d{2}(?:[ T].*)?$", na=False)
    dates.loc[iso] = pd.to_datetime(source_date.loc[iso], format="ISO8601", errors="coerce")
day = dates.dt.normalize()
summary.update({"positive_rows": int(label.eq(1).sum()), "legacy_zero_rows": int(label.eq(0).sum()),
 "missing_or_invalid_label_rows": int((~label.isin([0, 1])).sum()),
 "missing_date_rows": int(source_date.isna().sum()),
 "unparseable_date_rows": int((source_date.notna() & dates.isna()).sum()),
 "rows_with_time_component": int((dates.notna() & dates.ne(day)).sum()),
 "distinct_event_years": int(day.dt.year.nunique())})
for c in required[:3]:
    summary[c + "_missing_rows"] = int(df[c].isna().sum())
for c in required[1:3]:
    summary[c + "_non_10_digit_rows"] = int((df[c].notna() & ~df[c].str.fullmatch(r"\d{10}", na=False)).sum())
print(json.dumps(summary, indent=2))


## 4. Provisional positive-event grouping
Group patient + PCP + service day + specialist. Missing-key rows are quarantined, not merged. This can collapse real repeat visits on the same day. Source-row numbers below refer to zero-based imported data rows, not Excel row numbers. IDs are batch-local; persist the mapping before using it downstream.

In [ ]:
work = df.copy()
work["_service_day"] = day
work["_source_row"] = np.arange(len(work))
keys = ["mso_membership_id", "primary_care_provider_npi", "_service_day", "rendering_provider_npi"]
positive = work.loc[label.eq(1)].copy()
complete = positive[keys].notna().all(axis=1)
quarantine = positive.loc[~complete].copy()
eligible = positive.loc[complete].copy()
eligible["encounter_id"] = "ENC_" + eligible.groupby(keys, sort=True).ngroup().astype(str).str.zfill(10)
events = eligible[["encounter_id"] + keys].drop_duplicates().copy()
row_event_map = eligible[["_source_row", "encounter_id"]].copy()
group_sizes = eligible.groupby("encounter_id").size()
summary.update({"positive_rows_missing_event_key": len(quarantine), "provisional_events": len(events),
 "groups_with_multiple_source_rows": int(group_sizes.gt(1).sum()),
 "rows_collapsed_by_grouping": len(eligible) - len(events)})
if "principal_diagnosis_code" in eligible:
    diagnoses = eligible.groupby("encounter_id")["principal_diagnosis_code"].nunique()
    summary["events_with_multiple_diagnosis_codes"] = int(diagnoses.gt(1).sum())
    summary["events_without_diagnosis"] = int(diagnoses.eq(0).sum())
# No first-row selection for conflicting diagnoses or provider attributes.
assert len(row_event_map) == len(eligible)
assert events.encounter_id.is_unique
print(json.dumps(summary, indent=2))


## 5. Optional local files and what to report
Leave saving off until you review the checks. Files below contain private records and must stay local. Share only reviewed aggregate counts from `summary`, plus relevant coverage/parse counts. Do not send source rows, code-value lists, exact dates, or executed notebooks.

Next decisions after your report: confirm date format and missing tokens; reconcile source columns; inspect conflicting event diagnoses locally; then define chronological splits. History, encoders, imputation, and candidate sampling come after splits.

In [ ]:
if WRITE_LOCAL_FILES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    files = {"events_PRIVATE.csv": events, "row_event_map_PRIVATE.csv": row_event_map,
             "quarantine_PRIVATE.csv": quarantine}
    if any((OUTPUT_DIR / name).exists() for name in files):
        raise FileExistsError("Choose a new output directory to avoid overwriting prior private mappings.")
    for name, frame in files.items():
        frame.to_csv(OUTPUT_DIR / name, index=False)
print("First pass complete. Review aggregate counts before sharing.")


In [ ]:
# Header-only audit: prints column names, never record values.
import re
column_names = [str(c) for c in raw.columns]
print("ALL SOURCE COLUMN NAMES")
for name in column_names:
    print(repr(name))
print("\nPOSSIBLE SERVICE / SPECIALTY FIELDS (not automatic mappings)")
for name in column_names:
    if re.search(r"srvc|service|spec|prvdr.*type", name, flags=re.I):
        print(repr(name))


## 6. Second cleaning pass — aggregate diagnostics
Run the earlier loading, label/date, and encounter-grouping cells first. These cells do not change source data or apply mappings. Only insurance category labels are printed; diagnoses, IDs, and row-level values are not printed. Review small aggregate counts before sharing. Coverage uses positive rows and unique providers, not regression negatives.


In [ ]:
needed = ["df", "positive", "eligible", "events"]
if any(name not in globals() for name in needed):
    raise RuntimeError("Run the earlier loading and event-grouping cells first.")
print({"positive_rows": len(positive), "complete_positive_rows": len(eligible),
       "provisional_events": len(events)})


### Insurance: raw labels and mapping candidates
Retain the vetted meanings: IPA_CARRIER = insurance company; IPA_CARRIER_PRODUCT = ACO/NON-ACO. Case/whitespace normalization is diagnostic only. Do not infer non-ACO from an unknown value. Event counts may overlap when one event contains multiple labels; missing rows are shown separately.


In [ ]:
insurance_reports = {}
for col in ["IPA_CARRIER", "IPA_CARRIER_PRODUCT"]:
    if col not in positive:
        print(col, "absent"); continue
    counts = positive[col].value_counts(dropna=True).rename_axis("source_label").reset_index(name="positive_rows")
    counts["normalized_label_for_review"] = counts["source_label"].str.strip().str.casefold()
    if col in eligible:
        event_counts = eligible.dropna(subset=[col]).groupby(col)["encounter_id"].nunique()
        counts["distinct_events_with_label"] = counts["source_label"].map(event_counts).fillna(0).astype(int)
        conflicts = eligible.groupby("encounter_id")[col].nunique().gt(1).sum()
        print(col, {"events_with_multiple_labels": int(conflicts)})
    print(col, {"missing_positive_rows": int(positive[col].isna().sum())})
    insurance_reports[col] = counts
    display(counts)


### Diagnoses: preserve distinct codes, measure grouping ambiguity
An internal per-event tuple preserves every distinct nonmissing code, with no inferred ICD mapping or arbitrary first-code choice. This is a preprocessing representation, not a final encoder. Multiple codes may represent legitimate clinical detail or merged events; counts alone cannot distinguish these explanations. Procedure counts are diagnostic only and do not reinstate procedures as model inputs.


In [ ]:
diagnosis_summary = {}
if "principal_diagnosis_code" not in eligible:
    print("Diagnosis column absent")
else:
    event_diagnosis_codes = eligible.groupby("encounter_id")["principal_diagnosis_code"].agg(
        lambda s: tuple(sorted(s.dropna().unique())))
    code_counts = event_diagnosis_codes.map(len)
    distribution = code_counts.value_counts().sort_index().rename_axis("distinct_codes_per_event").reset_index(name="events")
    display(distribution)
    diagnosis_summary = {"events_without_codes": int(code_counts.eq(0).sum()),
                         "events_with_one_code": int(code_counts.eq(1).sum()),
                         "events_with_multiple_codes": int(code_counts.gt(1).sum())}
    group_checks = pd.DataFrame({"source_rows": eligible.groupby("encounter_id").size(),
                                "diagnosis_count": code_counts})
    if "procedure_code" in eligible:
        group_checks["procedure_count"] = eligible.groupby("encounter_id")["procedure_code"].nunique()
    rows=[]
    for name, mask in [("no diagnosis", code_counts.eq(0)), ("one diagnosis", code_counts.eq(1)),
                       ("multiple diagnoses", code_counts.gt(1))]:
        subset=group_checks.loc[mask]
        record={"group": name, "events": len(subset), "source_rows": int(subset.source_rows.sum()),
                "median_rows_per_event": float(subset.source_rows.median()) if len(subset) else None}
        if "procedure_count" in subset:
            record["events_with_multiple_procedures"] = int(subset.procedure_count.gt(1).sum())
        rows.append(record)
    display(pd.DataFrame(rows))
    print(diagnosis_summary)
# Do not display or export event_diagnosis_codes: its index contains encounter identifiers.


### Provider coverage and consistency
“Any value” means at least one nonmissing positive record for a provider, not complete historical coverage. Multiple values across years may be legitimate updates; within-year variation still needs interpretation. These diagnostics do not choose the first/latest record. Matching uses whitespace/case normalization only; disagreement can reflect different taxonomies or rating scales rather than an error.


In [ ]:
provider_fields = {
 "primary_care_provider_npi": ["Credential_PCP", "Rndrng_Prvdr_Crdntls_pcp", "practice size_pcp", "years in medicare_pcp", "Tot_Benes_pcp"],
 "rendering_provider_npi": ["clinician specialty", "rnd_pri_spec", "Credential_RP", "Rndrng_Prvdr_Crdntls",
 "Rating_RP", "Reviews_avg_rating_RP", "Number_of_ratings_RP", "final MIPS score",
 "practice size", "years in medicare", "Tot_Benes", "Tot_Srvcs", "Bene_Avg_Age", "Bene_Avg_Risk_Scre",
 "Bene_Age_LT_65_Cnt", "Bene_Age_65_74_Cnt", "Bene_Age_75_84_Cnt", "Bene_Age_GT_84_Cnt", "Bene_Feml_Cnt", "Bene_Male_Cnt"]}
provider_coverage=[]
for provider, fields in provider_fields.items():
    base = positive.loc[positive[provider].notna()].copy()
    base["_year"] = base["_service_day"].dt.year
    total = base[provider].nunique()
    for col in fields:
        if col not in base:
            provider_coverage.append({"role": provider, "column": col, "present": False}); continue
        unique_values=base.groupby(provider)[col].nunique()
        annual=base.dropna(subset=["_year"]).groupby([provider,"_year"])[col].nunique()
        provider_coverage.append({"role": provider, "column": col, "present": True,
          "positive_rows_missing": int(base[col].isna().sum()), "providers": int(total),
          "providers_without_any_value": int(unique_values.eq(0).sum()),
          "providers_with_multiple_values": int(unique_values.gt(1).sum()),
          "provider_years_with_multiple_values": int(annual.gt(1).sum())})
display(pd.DataFrame(provider_coverage))

pairs=[("primary_care_provider_npi", "Credential_PCP", "Rndrng_Prvdr_Crdntls_pcp"),
       ("rendering_provider_npi", "clinician specialty", "rnd_pri_spec"),
       ("rendering_provider_npi", "Credential_RP", "Rndrng_Prvdr_Crdntls"),
       ("rendering_provider_npi", "Rating_RP", "Reviews_avg_rating_RP")]
source_agreement=[]
for provider, left, right in pairs:
    if left not in positive or right not in positive: continue
    a=positive[left].str.strip().str.casefold(); b=positive[right].str.strip().str.casefold()
    both=a.notna() & b.notna()
    if left == "Rating_RP":
        an=pd.to_numeric(a,errors="coerce"); bn=pd.to_numeric(b,errors="coerce")
        comparable=both & an.notna() & bn.notna()
        equal=an.eq(bn).fillna(False) & comparable
    else:
        comparable=both; equal=a.eq(b).fillna(False) & comparable
    disagreement=comparable & ~equal
    source_agreement.append({"left":left,"right":right,
      "both_present_rows":int(both.sum()), "comparable_rows":int(comparable.sum()),
      "equal_rows":int(equal.sum()), "different_rows":int(disagreement.sum()),
      "left_only_rows":int((a.notna() & b.isna()).sum()),
      "right_only_rows":int((a.isna() & b.notna()).sum()),
      "neither_rows":int((a.isna() & b.isna()).sum()),
      "providers_with_any_disagreement":int(positive.loc[disagreement,provider].nunique())})
display(pd.DataFrame(source_agreement))


### What to share next
Save the notebook after running these new cells. The useful outputs are the insurance labels/counts, diagnosis-count distribution, and provider coverage/agreement summaries. No mapping, imputation, source substitution, or feature deletion has been applied by this second pass. Do not display the intermediate row-level frames or event diagnosis tuples.


## 7. Approved source choices and provider-conflict diagnostics
Run Sections 1–6 first. These cells create a separate working frame; source rows and earlier summaries remain unchanged. They do not export private data or choose one value for conflicting provider attributes. Outputs contain counts only.


In [ ]:
if "eligible" not in globals():
    raise RuntimeError("Run the earlier encounter-grouping cells first.")
selected = eligible.copy()
product_map = {"aco":"ACO", "ma":"NON-ACO", "tx aetna commercial":"NON-ACO",
               "anthem/pepsico":"NON-ACO", "texas health aetna":"NON-ACO"}
selected["aco_status"] = selected["IPA_CARRIER_PRODUCT"].str.strip().str.casefold().map(product_map).astype("string")
for role, primary, fallback in [("pcp", "Rndrng_Prvdr_Crdntls_pcp", "Credential_PCP"),
                                 ("specialist", "Rndrng_Prvdr_Crdntls", "Credential_RP")]:
    a = selected[primary] if primary in selected else pd.Series(pd.NA,index=selected.index,dtype="string")
    b = selected[fallback] if fallback in selected else pd.Series(pd.NA,index=selected.index,dtype="string")
    selected[role+"_credentials_selected"] = a.combine_first(b)
    origin = pd.Series("missing",index=selected.index,dtype="string")
    origin.loc[b.notna()] = fallback
    origin.loc[a.notna()] = primary
    selected[role+"_credentials_source"] = origin
selected["specialist_specialty_selected"] = selected["clinician specialty"]
# Preserve original numeric strings; reuse validated numeric parsing without filling missing values.
for output, source in [("specialist_rating_selected","Rating_RP"),
                       ("specialist_rating_count_selected","Number_of_ratings_RP")]:
    v=pd.to_numeric(selected[source],errors="coerce")
    selected[output]=v.where(np.isfinite(v.fillna(0).astype(float)))
# Distinct codes remain private preprocessing data; no first-code selection.
event_diagnosis_codes = selected.groupby("encounter_id")["principal_diagnosis_code"].agg(
    lambda s: tuple(sorted(s.dropna().unique())))
event_aco_counts = selected.groupby("encounter_id")["aco_status"].nunique()
event_aco = selected.groupby("encounter_id")["aco_status"].agg(
    lambda s: s.dropna().iloc[0] if s.dropna().nunique()==1 else pd.NA)
# Conflicting known statuses and all-unknown groups remain null; do not choose the first.
print({"rows_with_unknown_aco_status":int(selected.aco_status.isna().sum()),
       "events_with_conflicting_binary_status":int(event_aco_counts.gt(1).sum()),
       "events_with_no_known_binary_status":int(event_aco_counts.eq(0).sum()),
       "events_with_any_unknown_status_row":int(selected.loc[selected.aco_status.isna(),"encounter_id"].nunique())})
for role in ["pcp","specialist"]:
    print(role, selected[role+"_credentials_source"].value_counts().to_dict())


### Do participation or office fields separate conflicting values?
For each conflicting provider-year, test whether splitting its rows by a context field leaves a single target value in each subgroup. Missing context does not count as an explanation. This is association, not proof of cause or permission to select a value. ZIP fields are office proxies, not organization identifiers; a provider may have several offices. No provider IDs, ZIP values, participation labels, or individual scores are displayed.


In [ ]:
probe = selected.copy()
probe["_year"] = probe["_service_day"].dt.year
specs = [
 ("rendering_provider_npi","final MIPS score",["participation type","practice size","Provider_Zip5Texas","D_rnd_Zip"]),
 ("rendering_provider_npi","practice size",["participation type","Provider_Zip5Texas","D_rnd_Zip","Zip_1_RP"]),
 ("primary_care_provider_npi","practice size_pcp",["participation type_pcp","Provider_Zip5Texas_pcp","D_pcp_Zip","Zip_1_PCP_New"])]
conflict_results=[]
for provider,target,contexts in specs:
    if target not in probe: continue
    base=probe.dropna(subset=[provider,"_year"]).copy()
    # Compare numeric equality so 4 and 4.0 are not counted as distinct values.
    parsed=pd.to_numeric(base[target],errors="coerce")
    finite=parsed.notna() & np.isfinite(parsed.fillna(0).astype(float))
    invalid=int((base[target].notna() & ~finite).sum())
    base["_target"]=parsed.where(finite)
    base=base.dropna(subset=["_target"])
    key=[provider,"_year"]
    counts=base.groupby(key)["_target"].nunique()
    bad=counts[counts.gt(1)]
    index=pd.MultiIndex.from_frame(base[key])
    conflicting=base.loc[index.isin(bad.index)].copy()
    print(target,{"conflicting_provider_years_numeric":len(bad),"invalid_numeric_rows":invalid})
    for context in contexts:
        result={"target":target,"context":context,"present":context in base,
                "conflicting_provider_years":len(bad)}
        if context in base:
            g=conflicting.groupby(key)
            full=g[context].agg(lambda s:s.notna().all()).reindex(bad.index,fill_value=False)
            varied=g[context].nunique().reindex(bad.index,fill_value=0).gt(1)
            residual=conflicting.groupby(key+[context],dropna=False)["_target"].nunique()
            residual_max=residual.groupby(level=[0,1]).max().reindex(bad.index,fill_value=0)
            explained=full & varied & residual_max.le(1)
            result.update({"groups_with_incomplete_context":int((~full).sum()),
                           "groups_with_varying_context":int(varied.sum()),
                           "groups_separated_by_context":int(explained.sum()),
                           "groups_still_conflicting_within_context":int(residual_max.gt(1).sum())})
        conflict_results.append(result)
display(pd.DataFrame(conflict_results))


Save after running Section 7. We will review binary insurance conflicts and whether participation/office context separates provider values before defining any aggregation rule. No imputation, historical backfill, or final provider-node table has been created.


## 8. Assess the proposed chronological split
Run the encounter-grouping cells first. This diagnoses the first four observed years for training, fifth for validation, and sixth for testing. It does not train or finalize the split. Only years and aggregate counts are displayed, never identifiers or exact event dates. Missing calendar years and partial-year coverage are flagged. Review small counts before sharing.


In [ ]:
if "events" not in globals():
    raise RuntimeError("Run event grouping first.")
split_probe = events.copy()
if not split_probe["encounter_id"].is_unique:
    raise ValueError("Expected one row per reconstructed encounter.")
if split_probe["_service_day"].isna().any():
    raise ValueError("Resolve missing event dates before assessing splits.")
split_probe["_year"] = split_probe["_service_day"].dt.year.astype(int)
split_probe["_month"] = split_probe["_service_day"].dt.month
roles={"patients":"mso_membership_id", "pcps":"primary_care_provider_npi",
       "specialists":"rendering_provider_npi"}
yearly=[]
for year, group in split_probe.groupby("_year",sort=True):
    row={"year":int(year),"events":len(group),
         "event_share_pct":round(100*len(group)/len(split_probe),2),
         "months_with_events":int(group["_month"].nunique())}
    row.update({"unique_"+role:int(group[col].nunique()) for role,col in roles.items()})
    yearly.append(row)
yearly_report=pd.DataFrame(yearly)
display(yearly_report)
observed_years=sorted(split_probe["_year"].unique().tolist())
missing_years=sorted(set(range(observed_years[0], observed_years[-1]+1))-set(observed_years)) if observed_years else []
print({"observed_years":observed_years,"missing_calendar_years":missing_years})
if len(observed_years)!=6:
    raise ValueError("The proposed 4/1/1 split assumes six observed years. Review coverage before choosing boundaries.")
year_to_split={y:"train" for y in observed_years[:4]}
year_to_split.update({observed_years[4]:"validation", observed_years[5]:"test"})
split_probe["proposed_split"]=split_probe["_year"].map(year_to_split)
split_report=[]
for name in ["train","validation","test"]:
    group=split_probe.loc[split_probe.proposed_split.eq(name)]
    row={"split":name,"years":", ".join(map(str,sorted(group._year.unique()))),
         "events":len(group),"event_share_pct":round(100*len(group)/len(split_probe),2)}
    row.update({"unique_"+role:int(group[col].nunique()) for role,col in roles.items()})
    split_report.append(row)
display(pd.DataFrame(split_report))
assert split_probe.groupby("_service_day")["proposed_split"].nunique().max()==1


### Unseen entities and referral pairs
The primary comparison is against training only. Additional test-versus-training-plus-validation counts show which entities could be known from accumulated history, without assuming model retraining. An unseen specialist may need a feature-based representation and an eligible candidate record. These counts alone do not establish candidate availability at prediction time.


In [ ]:
train_probe=split_probe.loc[split_probe.proposed_split.eq("train")]
validation_probe=split_probe.loc[split_probe.proposed_split.eq("validation")]
test_probe=split_probe.loc[split_probe.proposed_split.eq("test")]
comparisons=[("validation","training",validation_probe,train_probe),
             ("test","training",test_probe,train_probe),
             ("test","training + validation",test_probe,pd.concat([train_probe,validation_probe]))]
unseen_report=[]
combination_report=[]
for target_name,reference_name,target,reference in comparisons:
    event_unseen={}
    for role,col in roles.items():
        known=set(reference[col].dropna())
        unique=set(target[col].dropna())
        mask=~target[col].isin(known)
        event_unseen[role]=mask
        unseen_report.append({"split":target_name,"reference":reference_name,"entity":role,
          "unique_entities":len(unique),"unseen_unique_entities":len(unique-known),
          "unseen_unique_pct":round(100*len(unique-known)/len(unique),2) if unique else None,
          "events_with_unseen_entity":int(mask.sum()),
          "event_pct":round(100*mask.mean(),2) if len(target) else None})
    pair_cols=["primary_care_provider_npi","rendering_provider_npi"]
    known_pairs=pd.MultiIndex.from_frame(reference[pair_cols].drop_duplicates())
    new_pairs=~pd.MultiIndex.from_frame(target[pair_cols]).isin(known_pairs)
    all_known=~(event_unseen["patients"] | event_unseen["pcps"] | event_unseen["specialists"])
    combination_report.append({"split":target_name,"reference":reference_name,
        "events_all_three_entities_known":int(all_known.sum()),
        "events_any_entity_unseen":int((~all_known).sum()),
        "events_with_unseen_pcp_specialist_pair":int(new_pairs.sum()),
        "unseen_pair_event_pct":round(100*new_pairs.mean(),2) if len(target) else None})
display(pd.DataFrame(unseen_report))
display(pd.DataFrame(combination_report))
print("Diagnostic only: no final split files written and no preprocessing fitted.")


Save these outputs for review. We will check event balance, calendar coverage, and unseen-entity rates before accepting the split. If the final year is sparse or partial, we can adjust chronological boundaries; we will not mix future events into training to improve coverage.


## 9. Final chronological assignments and strictly prior pair history
Approved: train through 2017, validation 2018, test observed 2019. This uses rolling observed history: a completed event can inform a later date, including during validation/test; model weights are not updated by this operation. Same-day events never inform one another. Original records remain untouched and no private files are automatically exported.

Features below are attached to observed positive events for preparation. During ranking the same lookup must be called for **every candidate**, not just the true specialist. Candidate selection, distances, graph visibility, and provider-attribute conflict resolution are still separate work.


In [ ]:
if "events" not in globals():
    raise RuntimeError("Run encounter construction first.")
history_events = events.copy()
if history_events.empty or not history_events.encounter_id.is_unique:
    raise ValueError("Expected nonempty unique encounters.")
if history_events[["_service_day", "primary_care_provider_npi", "rendering_provider_npi"]].isna().any().any():
    raise ValueError("History requires complete provider keys and dates.")
years = history_events._service_day.dt.year
if not years.between(2014,2019).all():
    raise ValueError("Unexpected event years; review the approved boundaries before proceeding.")
history_events["split"] = np.select([years.le(2017),years.eq(2018),years.eq(2019)],
                                    ["train","validation","test"],default="unassigned")
assert history_events.groupby("_service_day")["split"].nunique().max()==1
split_assignments = history_events[["encounter_id", "split"]].copy()  # private IDs, do not display
display(history_events.groupby("split").size().reindex(["train","validation","test"]).rename("events").reset_index())

PAIR = ["primary_care_provider_npi", "rendering_provider_npi"]
pair_daily = (history_events.groupby(PAIR+["_service_day"]).size()
              .rename("events_on_day").reset_index().sort_values(PAIR+["_service_day"]))
pair_daily["cumulative_events"] = pair_daily.groupby(PAIR)["events_on_day"].cumsum()
pair_daily["first_pair_date"] = pair_daily.groupby(PAIR)["_service_day"].transform("min")
# Full history is stored for lookup, but each query uses a strictly backward join.
def lookup_prior_referrals(queries, daily_history):
    """Return candidate pair features in input row order. No exact-date matches allowed."""
    required=PAIR+["_service_day"]
    if queries[required].isna().any().any():
        raise ValueError("Candidate queries require complete PCP, specialist and date keys.")
    q=queries[required].copy().reset_index(drop=True)
    q["_order"]=np.arange(len(q))
    if len(q)==0:
        return pd.DataFrame({"prior_referral_count":pd.Series(dtype="int64"),
          "first_observed_referral_date":pd.Series(dtype="datetime64[ns]"),
          "days_since_last_referral":pd.Series(dtype="Int64")})
    h=daily_history[PAIR+["_service_day","cumulative_events","first_pair_date"]].copy()
    h=h.rename(columns={"_service_day":"_previous_day"})
    # pandas requires identical merge-key dtypes, including string NA semantics.
    # Normalize both inputs; preserve identifiers as text, never numeric.
    key_dtype = pd.StringDtype(storage="python")
    for key in PAIR:
        q[key] = q[key].astype(key_dtype)
        h[key] = h[key].astype(key_dtype)
    q["_service_day"] = pd.to_datetime(q["_service_day"]).astype("datetime64[ns]")
    h["_previous_day"] = pd.to_datetime(h["_previous_day"]).astype("datetime64[ns]")
    joined=pd.merge_asof(q.sort_values("_service_day"), h.sort_values("_previous_day"),
                        left_on="_service_day",right_on="_previous_day",by=PAIR,
                        direction="backward",allow_exact_matches=False).sort_values("_order")
    prior=joined["cumulative_events"].fillna(0).astype("int64")
    recency=(joined["_service_day"]-joined["_previous_day"]).dt.days.astype("Int64")
    assert (recency.dropna()>0).all()
    result=pd.DataFrame({"prior_referral_count":prior.to_numpy(),
       "first_observed_referral_date":joined["first_pair_date"].to_numpy(),
       "days_since_last_referral":recency.array})
    assert result.loc[result.prior_referral_count.eq(0),"first_observed_referral_date"].isna().all()
    return result

history_features = lookup_prior_referrals(history_events, pair_daily)
events_with_history = pd.concat([history_events.reset_index(drop=True),history_features],axis=1)
# Do not display events_with_history or pair_daily; both contain private identifiers/dates.


### Aggregate history coverage and checks
Frequency counts distinct grouped encounters, not source claim rows or distinct patients. First observed date describes the available history, not first-ever referral. A zero frequency has missing recency. The history table is supporting preprocessing, not a direct PCP–Specialist graph edge.


In [ ]:
report=[]
for name in ["train","validation","test"]:
    part=events_with_history.loc[events_with_history.split.eq(name)]
    seen=part.prior_referral_count.gt(0)
    report.append({"split":name,"events":len(part),"events_with_prior_pair_history":int(seen.sum()),
      "events_without_prior_pair_history":int((~seen).sum()),
      "prior_history_pct":round(100*seen.mean(),2) if len(part) else None,
      "median_prior_referral_count":float(part.prior_referral_count.median()) if len(part) else None,
      "median_recency_days_when_known":float(part.loc[seen,"days_since_last_referral"].median()) if seen.any() else None})
display(pd.DataFrame(report))
same_day_consistency=events_with_history.groupby(PAIR+["_service_day"])["prior_referral_count"].nunique()
assert same_day_consistency.le(1).all()
assert len(events_with_history)==len(events)
assert events_with_history.encounter_id.is_unique
print({"same_day_history_consistent":True,"event_count_preserved":True,
       "same_pair_days_with_multiple_events":int(pair_daily.events_on_day.gt(1).sum())})
print("Split assignments and history created in memory. No model trained or private files exported.")


## 10. Candidate availability and training comparisons
Initial policy: a specialist becomes a candidate after their first observed positive event date, across any PCP. Same-day first appearances are excluded. This approximates an available provider directory; it does not prove eligibility, clinical fit, insurance acceptance, or appointment availability. No filtering uses the target specialist's specialty. We can add clinical/access restrictions after validating their inputs.

Training samples up to 20 other available specialists uniformly without replacement (fixed seed). These are unobserved comparisons, not proven inappropriate referrals. Validation/test use all available candidates, in lazy batches to avoid an enormous cross-product. Targets outside the pool are reported explicitly. Headline all-query ranking must count unavailable targets as misses; report conditional ranking on eligible queries separately.

This section supplies temporal pair features. Candidate distances and provider attributes are not yet attached; training is not ready until these are addressed. Do not copy the observed target's distances to alternatives.


In [ ]:
if "events_with_history" not in globals() or "lookup_prior_referrals" not in globals():
    raise RuntimeError("Run Section 9 first.")
NEGATIVES_PER_QUERY = 20
CANDIDATE_SEED = 42
event_candidates_source=events_with_history.reset_index(drop=True).copy()
specialist_first_seen=(event_candidates_source.groupby("rendering_provider_npi")["_service_day"]
                       .min().sort_values(kind="stable"))
candidate_ids=specialist_first_seen.index.to_numpy()
candidate_start_dates=specialist_first_seen.to_numpy(dtype="datetime64[ns]")
def available_specialists(query_date):
    cutoff=np.searchsorted(candidate_start_dates,np.datetime64(query_date,"ns"),side="left")
    return candidate_ids[:cutoff]

pool_sizes=np.searchsorted(candidate_start_dates,
    event_candidates_source._service_day.to_numpy(dtype="datetime64[ns]"),side="left")
target_first=event_candidates_source.rendering_provider_npi.map(specialist_first_seen)
target_available=target_first.lt(event_candidates_source._service_day)
candidate_audit=event_candidates_source[["encounter_id","split"]].copy()
candidate_audit["candidate_count"]=pool_sizes
candidate_audit["target_available"]=target_available.to_numpy()
reports=[]
for name in ["train","validation","test"]:
    a=candidate_audit.loc[candidate_audit.split.eq(name)]
    reports.append({"split":name,"queries":len(a),
      "queries_with_empty_pool":int(a.candidate_count.eq(0).sum()),
      "queries_target_unavailable":int((~a.target_available).sum()),
      "target_available_pct":round(100*a.target_available.mean(),2) if len(a) else None,
      "median_candidates":float(a.candidate_count.median()) if len(a) else None,
      "max_candidates":int(a.candidate_count.max()) if len(a) else 0,
      "full_ranking_pairs":int(a.candidate_count.sum())})
display(pd.DataFrame(reports))


### Training rows and lazy evaluation batches
Only training queries whose observed target is available and that have at least one alternative are included in this initial sampled dataset. The audit preserves counts of all excluded queries. During evaluation no true target is forced into the pool. Targets are labels only; candidate NPI selects the scored specialist, not a query feature.


In [ ]:
rng=np.random.default_rng(CANDIDATE_SEED)
records=[]
train_query_counts={"eligible_queries":0,"excluded_target_unavailable":0,"excluded_no_alternative":0}
for row in event_candidates_source.loc[event_candidates_source.split.eq("train")].itertuples(index=False, name=None):
    # Explicit column lookup avoids namedtuple renaming of leading-underscore columns.
    record=dict(zip(event_candidates_source.columns,row))
    pool=available_specialists(record["_service_day"])
    target=record["rendering_provider_npi"]
    if not (pool==target).any():
        train_query_counts["excluded_target_unavailable"]+=1; continue
    alternatives=pool[pool!=target]
    if not len(alternatives):
        train_query_counts["excluded_no_alternative"]+=1; continue
    sampled=rng.choice(alternatives,size=min(NEGATIVES_PER_QUERY,len(alternatives)),replace=False)
    train_query_counts["eligible_queries"]+=1
    for candidate in np.concatenate(([target],sampled)):
        records.append((record["encounter_id"],record["primary_care_provider_npi"],
                        record["_service_day"],candidate,int(candidate==target)))
candidate_columns=["encounter_id","primary_care_provider_npi","_service_day","rendering_provider_npi","label"]
train_candidates=pd.DataFrame.from_records(records,columns=candidate_columns)
del records
train_candidates["_service_day"]=pd.to_datetime(train_candidates["_service_day"])
if len(train_candidates):
    features=lookup_prior_referrals(train_candidates,pair_daily)
    train_candidates=pd.concat([train_candidates.reset_index(drop=True),features],axis=1)
    # Shuffle so positive-first construction cannot influence downstream batching/order.
    train_candidates=train_candidates.sample(frac=1,random_state=CANDIDATE_SEED).reset_index(drop=True)
    assert not train_candidates.duplicated(["encounter_id","rendering_provider_npi"]).any()
    assert train_candidates.groupby("encounter_id").label.sum().eq(1).all()
print({**train_query_counts,"training_candidate_rows":len(train_candidates),
       "positive_rows":int(train_candidates.label.sum())})

def iter_evaluation_candidates(split_name, chunk_size=10000):
    """Yield private candidate frames for full ranking; do not display frames."""
    if split_name not in {"validation","test"} or chunk_size<1:
        raise ValueError("Choose validation/test and positive chunk_size.")
    source=event_candidates_source.loc[event_candidates_source.split.eq(split_name)]
    for eid,pcp,date,target in source[["encounter_id","primary_care_provider_npi","_service_day","rendering_provider_npi"]].itertuples(index=False,name=None):
        pool=available_specialists(date)
        for start in range(0,len(pool),chunk_size):
            candidates=pool[start:start+chunk_size]
            batch=pd.DataFrame({"encounter_id":eid,"primary_care_provider_npi":pcp,
              "_service_day":date,"rendering_provider_npi":candidates,"label":(candidates==target).astype(int)})
            features=lookup_prior_referrals(batch,pair_daily)
            yield pd.concat([batch.reset_index(drop=True),features],axis=1)
# Consume just one batch to check compatibility, not the full evaluation cross-product.
probe_batch=next(iter_evaluation_candidates("validation"),None)
print({"evaluation_batch_smoke_check":probe_batch is not None,
       "probe_rows":len(probe_batch) if probe_batch is not None else 0})
del probe_batch
print("Candidate tables are private in-memory objects. No files exported and no model trained.")


Save the aggregate outputs for review. Candidate-pool coverage is a limit on this initial historical-directory policy, not a measure of the model. New specialists could be recommended earlier if we later obtain independently dated directory records. Before training: supply candidate-level distances consistently, resolve provider attributes, and ensure graph history uses the same strict cutoff. In full-ranking evaluation combine all chunks of a query before calculating ranks; empty-pool and unavailable-target queries remain in the all-query denominator.


## 11. Candidate-distance coverage without target-row leakage
Run Sections 9–10 first. This audit tests whether distances from **strictly earlier positive records** can supply candidate pairs. Patient distance uses patient–specialist history; PCP distance uses PCP–specialist history. The same lookup applies to targets and alternatives. No target-event distance, regression-negative distance, ZIP geocoding, or future record is used.

Historical reuse is only a provisional coverage diagnostic: patients/providers can move and office choice can differ. Same-pair/same-day conflicting distances invalidate that day's observation rather than choosing the first. A later missing/conflicting observation also makes the lookup unavailable until another valid observation. No distance features are added to the training table by this audit.


In [ ]:
if "train_candidates" not in globals() or "eligible" not in globals():
    raise RuntimeError("Run encounter construction and Sections 9–10 first.")
distance_specs={
 "patient_specialist_distance":(["mso_membership_id","rendering_provider_npi"],"D_pat_rnd_dist"),
 "pcp_specialist_distance":(["primary_care_provider_npi","rendering_provider_npi"],"D_pcp_rnd_dist")}
distance_histories={}
distance_source_report=[]
for name,(keys,col) in distance_specs.items():
    if col not in eligible:
        print(name,"source column absent"); continue
    base=eligible[keys+["_service_day",col]].copy()
    v=pd.to_numeric(base[col],errors="coerce")
    valid=v.notna() & np.isfinite(v.fillna(0).astype(float)) & v.ge(0)
    base["_distance"]=v.where(valid)
    grouped=base.groupby(keys+["_service_day"])["_distance"]
    daily=grouped.agg(["nunique","first"]).reset_index()
    daily["distance"]=daily["first"].where(daily["nunique"].eq(1))
    daily=daily.rename(columns={"_service_day":"_distance_observed_day"})
    distance_histories[name]=daily[keys+["_distance_observed_day","distance"]]
    distance_source_report.append({"distance":name,"source_rows":len(base),
      "missing_or_invalid_rows":int((~valid).sum()),"pair_days":len(daily),
      "pair_days_conflicting_values":int(daily["nunique"].gt(1).sum()),
      "pair_days_no_valid_value":int(daily["nunique"].eq(0).sum())})
display(pd.DataFrame(distance_source_report))

def prior_distance_lookup(queries,name):
    keys,_=distance_specs[name]
    q=queries[keys+["_service_day"]].copy().reset_index(drop=True)
    if q.isna().any().any():
        raise ValueError("Distance queries require complete pair identifiers and date.")
    q["_order"]=np.arange(len(q))
    h=distance_histories[name].copy()
    dtype=pd.StringDtype(storage="python")
    for key in keys:
        q[key]=q[key].astype(dtype); h[key]=h[key].astype(dtype)
    q["_service_day"]=pd.to_datetime(q["_service_day"]).astype("datetime64[ns]")
    h["_distance_observed_day"]=pd.to_datetime(h["_distance_observed_day"]).astype("datetime64[ns]")
    if q.empty:
        return pd.DataFrame({"distance":pd.Series(dtype="float64"),"age_days":pd.Series(dtype="float64")})
    merged=pd.merge_asof(q.sort_values("_service_day"),h.sort_values("_distance_observed_day"),
                        left_on="_service_day",right_on="_distance_observed_day",by=keys,
                        direction="backward",allow_exact_matches=False).sort_values("_order")
    age=(merged._service_day-merged._distance_observed_day).dt.days
    assert age.dropna().gt(0).all()
    return pd.DataFrame({"distance":merged.distance.to_numpy(),"age_days":age.to_numpy()})


### Coverage on a bounded diagnostic sample
Use up to 2,000 reproducibly sampled encounters per split with an available target and at least one alternative; retain the target and up to 20 uniformly sampled alternatives for each. This is a diagnostic sample, not full-ranking evaluation. Training examples are resampled here without changing train_candidates. Excluded unavailable targets remain counted by Section 10.

Different coverage between targets and alternatives is a warning that distance availability tracks historical selection. Even symmetric prior-only lookup is not an independent accessibility measure. Report results before deciding on geocoding, imputation, or distance inclusion.


In [ ]:
DISTANCE_AUDIT_QUERIES_PER_SPLIT=2000
DISTANCE_AUDIT_ALTERNATIVES=20
audit_rng=np.random.default_rng(43)
audit_records=[]
audit_query_report=[]
for split in ["train","validation","test"]:
    mask=event_candidates_source.split.eq(split) & candidate_audit.target_available & candidate_audit.candidate_count.gt(1)
    pool=event_candidates_source.loc[mask]
    sample=pool.sample(n=min(len(pool),DISTANCE_AUDIT_QUERIES_PER_SPLIT),random_state=43)
    audit_query_report.append({"split":split,"eligible_queries":len(pool),"sampled_queries":len(sample)})
    for eid,patient,pcp,date,target in sample[["encounter_id","mso_membership_id","primary_care_provider_npi","_service_day","rendering_provider_npi"]].itertuples(index=False,name=None):
        alternatives=available_specialists(date)
        alternatives=alternatives[alternatives!=target]
        chosen=audit_rng.choice(alternatives,size=min(len(alternatives),DISTANCE_AUDIT_ALTERNATIVES),replace=False)
        for specialist in np.concatenate(([target],chosen)):
            audit_records.append((split,eid,patient,pcp,date,specialist,int(specialist==target)))
distance_queries=pd.DataFrame(audit_records,columns=["split","encounter_id","mso_membership_id",
    "primary_care_provider_npi","_service_day","rendering_provider_npi","label"])
del audit_records
display(pd.DataFrame(audit_query_report))
coverage=[]
for name in distance_histories:
    found=prior_distance_lookup(distance_queries,name)
    for split in ["train","validation","test"]:
        for label,kind in [(1,"observed target"),(0,"sampled alternative")]:
            mask=distance_queries.split.eq(split) & distance_queries.label.eq(label)
            values=found.loc[mask]
            known=values.distance.notna()
            coverage.append({"split":split,"candidate_kind":kind,"distance":name,
              "candidate_rows":len(values),"distance_available_rows":int(known.sum()),
              "coverage_pct":round(100*known.mean(),2) if len(values) else None,
              "median_observation_age_days":float(values.loc[known,"age_days"].median()) if known.any() else None})
display(pd.DataFrame(coverage))
print("Audit only: historical distances not attached to model inputs; no raw records or files exported.")


## 12. Location metadata for candidate-wide distances
Distance does not require a previous referral. We need a patient/PCP location and a candidate office location, then a ZIP-to-coordinate reference. This section audits existing columns only: no external lookup, no geocoding, and no patient ZIP inferred from distances. ZIP metadata stays separate from predictive inputs.

Run the loading, positive-record, and grouping cells first. ZIP values and identifiers are never displayed. Valid format is not proof of a real ZIP, correct office, or historical availability. Coverage is measured over the observed positive-record population, not a dated provider directory.


In [ ]:
import re
if "positive" not in globals():
    raise RuntimeError("Run the earlier loading and event-grouping cells first.")
location_headers=[str(c) for c in df.columns if re.search(r"zip|postal|address|latitude|longitude|city|county|location",str(c),re.I)]
print("Location-related column names only:")
for name in location_headers: print(name)
patient_named_headers=[c for c in location_headers if re.search(r"patient|pat_|member|mso|home|residen",c,re.I)]
print("Patient-named location candidates (names do not establish meaning):",patient_named_headers)
if not patient_named_headers:
    print("No explicitly patient-named location column found. Do not substitute provider or medical-school ZIPs.")

def normalize_zip(series):
    raw=series.astype("string").str.strip()
    # Repair only unambiguous spreadsheet integer formatting, never infer missing leading zeros.
    candidate=raw.str.replace(r"^(\d{5})\.0$",r"\1",regex=True)
    five=candidate.str.fullmatch(r"\d{5}",na=False)
    plus4=candidate.str.fullmatch(r"\d{5}-\d{4}|\d{9}",na=False)
    result=candidate.where(five).combine_first(candidate.str.slice(0,5).where(plus4))
    result=result.mask(result.eq("00000"))
    return result.astype("string")

zip_roles={
 "pcp":("primary_care_provider_npi",["Provider_Zip5Texas_pcp","D_pcp_Zip","Zip_1_PCP_New","Zip_2_PCP_New","Zip_3_PCP_New"]),
 "specialist":("rendering_provider_npi",["Provider_Zip5Texas","D_rnd_Zip","Zip_1_RP","Zip_2_RP","Zip_3_RP"])}
normalized_zips={}
zip_coverage=[]
for role,(provider,columns) in zip_roles.items():
    for col in columns:
        if col not in positive:
            zip_coverage.append({"role":role,"column":col,"present":False}); continue
        z=normalize_zip(positive[col]); normalized_zips[col]=z
        base=pd.DataFrame({"provider":positive[provider],"year":positive._service_day.dt.year,"zip":z})
        by_provider=base.groupby("provider")["zip"].nunique()
        by_year=base.dropna(subset=["year"]).groupby(["provider","year"])["zip"].nunique()
        zip_coverage.append({"role":role,"column":col,"present":True,
          "positive_rows":len(base),"raw_missing_rows":int(positive[col].isna().sum()),
          "nonmissing_invalid_or_ambiguous_rows":int((positive[col].notna() & z.isna()).sum()),
          "valid_format_rows":int(z.notna().sum()),"providers":len(by_provider),
          "providers_without_valid_zip":int(by_provider.eq(0).sum()),
          "providers_with_multiple_zips":int(by_provider.gt(1).sum()),
          "provider_years_with_multiple_zips":int(by_year.gt(1).sum())})
display(pd.DataFrame(zip_coverage))


### Compare office sources and multiple-office coverage
Rows with multiple distinct office ZIPs need an explicit location policy. Multiple offices are not automatically data errors. Derived ZIP agreement may reveal which office source was used, but agreement alone does not prove provenance. Medical-school ZIPs are deliberately excluded.


In [ ]:
comparisons=[]
for role,(provider,columns) in zip_roles.items():
    primary,derived,office1,*_=columns
    for left,right in [(primary,derived),(primary,office1),(derived,office1)]:
        if left not in normalized_zips or right not in normalized_zips: continue
        a=normalized_zips[left]; b=normalized_zips[right]
        both=a.notna() & b.notna(); different=both & a.ne(b).fillna(False)
        comparisons.append({"role":role,"left":left,"right":right,
          "both_valid_rows":int(both.sum()),"different_rows":int(different.sum()),
          "providers_with_any_disagreement":int(positive.loc[different,provider].nunique()),
          "left_only_valid_rows":int((a.notna() & b.isna()).sum()),
          "right_only_valid_rows":int((a.isna() & b.notna()).sum())})
display(pd.DataFrame(comparisons))
office_reports=[]
for role,(provider,columns) in zip_roles.items():
    present=[c for c in columns if c in normalized_zips]
    if not present: continue
    zips=pd.DataFrame({c:normalized_zips[c] for c in present})
    distinct=zips.nunique(axis=1,dropna=True)
    provider_has=distinct.gt(0).groupby(positive[provider]).any()
    # Union over all observed positive records; diagnostic only, not a historical location lookup.
    office_reports.append({"role":role,"providers":len(provider_has),
        "providers_with_no_valid_office_zip":int((~provider_has).sum()),
        "rows_with_no_valid_office_zip":int(distinct.eq(0).sum()),
        "rows_with_multiple_distinct_office_zips":int(distinct.gt(1).sum())})
display(pd.DataFrame(office_reports))
print("Location audit only: no source selected, ZIP coordinates downloaded, or candidate distances calculated.")


Next: choose provider location sources after reviewing these summaries, and identify the patient-location source outside this table if needed. If patient ZIP is in a separate local file, join it locally using a verified patient ID/date mapping; never print patient ZIP pairs. Do not assume office data collected after an event was historically available. With source assumptions documented, a local ZIP-coordinate reference can supply distances for all candidate pairs, irrespective of interaction history.


## 13. PCP–specialist geographic distance for any candidate
Sources: `Provider_Zip5Texas_pcp` and `Provider_Zip5Texas`, provisionally treated as fixed office snapshots. Distances use great-circle miles between Census 2020 ZCTA representative points (`INTPTLAT`, `INTPTLONG`), not exact office or driving distances. A ZCTA is an approximation of a ZIP area; some ZIPs have no match. Unmatched ZIPs remain missing, never zero. Same matched ZIP gives zero approximate distance.

Reference: https://www.census.gov/geographies/reference-files/2020/geo/gazetter-file.html
The 2020 geography is a fixed approximation applied to 2014–2019 events, not a historically versioned location dataset. Provider candidates still follow the strictly earlier availability rule. This step does not infer patient locations or implement patient distance.

The first cell downloads a public reference once. No private ZIP, identifier, or record is sent: it fetches the same national file for every user. Offline option: download that file separately to REFERENCE_PATH and set DOWNLOAD_PUBLIC_REFERENCE=False.


In [ ]:
import zipfile, urllib.request, hashlib
REFERENCE_URL="https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_Gaz_zcta_national.zip"
REFERENCE_PATH=Path("reference_data/2020_Gaz_zcta_national.zip")
DOWNLOAD_PUBLIC_REFERENCE=True
if not REFERENCE_PATH.exists():
    if not DOWNLOAD_PUBLIC_REFERENCE:
        raise FileNotFoundError("Place the public Census archive at REFERENCE_PATH.")
    with urllib.request.urlopen(REFERENCE_URL,timeout=60) as response:
        public_bytes=response.read()
    # Verify archive before saving; no private parameters are included in request.
    import io
    with zipfile.ZipFile(io.BytesIO(public_bytes)) as archive:
        if archive.testzip() is not None: raise ValueError("Corrupt reference archive")
    REFERENCE_PATH.parent.mkdir(parents=True,exist_ok=True)
    REFERENCE_PATH.write_bytes(public_bytes)
with zipfile.ZipFile(REFERENCE_PATH) as archive:
    names=[x for x in archive.namelist() if x.lower().endswith(".txt")]
    if len(names)!=1: raise ValueError("Unexpected reference archive structure")
    with archive.open(names[0]) as stream:
        zcta=pd.read_csv(stream,sep="\t",dtype="string",keep_default_na=False)
zcta.columns=zcta.columns.str.strip()
zcta["GEOID"]=zcta.GEOID.str.strip()
for col in ["INTPTLAT","INTPTLONG"]:
    zcta[col]=pd.to_numeric(zcta[col].str.strip(),errors="coerce")
if not zcta.GEOID.is_unique: raise ValueError("Duplicate ZCTA keys")
if not (zcta.INTPTLAT.between(-90,90) & zcta.INTPTLONG.between(-180,180)).all():
    raise ValueError("Invalid reference coordinates")
zip_coordinates=zcta.set_index("GEOID")[["INTPTLAT","INTPTLONG"]]
print({"public_reference_rows":len(zcta),"reference_sha256":hashlib.sha256(REFERENCE_PATH.read_bytes()).hexdigest()})


In [ ]:
if "normalize_zip" not in globals() or "train_candidates" not in globals():
    raise RuntimeError("Run candidate construction and Section 12 first.")
provider_location_maps={}
location_reports=[]
for role,id_col,zip_col in [("pcp","primary_care_provider_npi","Provider_Zip5Texas_pcp"),
                            ("specialist","rendering_provider_npi","Provider_Zip5Texas")]:
    base=pd.DataFrame({"provider":positive[id_col],"zip":normalize_zip(positive[zip_col])})
    counts=base.groupby("provider")["zip"].nunique()
    if counts.gt(1).any():
        raise ValueError("Chosen office source now has multiple ZIPs per provider; review rather than selecting first.")
    mapping=base.dropna().drop_duplicates("provider").set_index("provider")["zip"]
    coords=pd.DataFrame({"lat":mapping.map(zip_coordinates.INTPTLAT),
                         "lon":mapping.map(zip_coordinates.INTPTLONG)})
    provider_location_maps[role]=coords
    location_reports.append({"role":role,"providers":len(counts),
       "providers_with_valid_zip":len(mapping),"providers_with_coordinates":int(coords.notna().all(axis=1).sum()),
       "providers_without_coordinates":int(len(counts)-coords.notna().all(axis=1).sum())})
display(pd.DataFrame(location_reports))

def geographic_pcp_distance(candidates):
    """Miles, returned in candidate row order; never use the observed referral's distance."""
    p=provider_location_maps["pcp"]; s=provider_location_maps["specialist"]
    lat1=candidates.primary_care_provider_npi.map(p.lat).to_numpy(dtype=float,na_value=np.nan)
    lon1=candidates.primary_care_provider_npi.map(p.lon).to_numpy(dtype=float,na_value=np.nan)
    lat2=candidates.rendering_provider_npi.map(s.lat).to_numpy(dtype=float,na_value=np.nan)
    lon2=candidates.rendering_provider_npi.map(s.lon).to_numpy(dtype=float,na_value=np.nan)
    lat1,lon1,lat2,lon2=map(np.radians,(lat1,lon1,lat2,lon2))
    a=np.sin((lat2-lat1)/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    miles=3958.7613*2*np.arcsin(np.sqrt(np.clip(a,0,1)))
    return pd.Series(miles,index=candidates.index,dtype="float64")

# Use a new explicit name; the original source distance remains untouched.
train_candidates["pcp_specialist_geo_distance_miles"]=geographic_pcp_distance(train_candidates)
train_candidates["pcp_specialist_geo_distance_missing"]=train_candidates.pcp_specialist_geo_distance_miles.isna()
geo_reports=[]
for label,kind in [(1,"observed target"),(0,"sampled alternative")]:
    v=train_candidates.loc[train_candidates.label.eq(label),"pcp_specialist_geo_distance_miles"]
    geo_reports.append({"split":"train","kind":kind,"candidate_rows":len(v),
      "coverage_pct":round(100*v.notna().mean(),2) if len(v) else None})
if "distance_queries" in globals():
    audit_distance=geographic_pcp_distance(distance_queries)
    for split in ["validation","test"]:
        for label,kind in [(1,"observed target"),(0,"sampled alternative")]:
            mask=distance_queries.split.eq(split) & distance_queries.label.eq(label)
            v=audit_distance.loc[mask]
            geo_reports.append({"split":split,"kind":kind,"candidate_rows":len(v),
              "coverage_pct":round(100*v.notna().mean(),2) if len(v) else None})
display(pd.DataFrame(geo_reports))

def iter_evaluation_candidates_with_distance(split_name,chunk_size=10000):
    for batch in iter_evaluation_candidates(split_name,chunk_size):
        batch["pcp_specialist_geo_distance_miles"]=geographic_pcp_distance(batch)
        batch["pcp_specialist_geo_distance_missing"]=batch.pcp_specialist_geo_distance_miles.isna()
        yield batch
print("Geographic distance attached to training candidates; use iter_evaluation_candidates_with_distance for ranking. No private files exported.")


## 14. Conservative provider histories and feature tables
Run Sections 7, 9, 10, and 13 first. No encoders or scalers are fitted here. IDs are linkage only. Provider features use records strictly before each query date. A numeric field is usable only when all valid observations so far agree; conflicting fields become missing with an ambiguity flag. Conflict does not remove a provider or the feature. This deliberately conservative baseline does not infer which group/practice score is appropriate or resolve genuine changes over time. No full-year aggregates or future rows resolve an earlier query.

Source values may still originate from fixed later snapshots, as previously documented. A strict observation cutoff cannot repair that source-provenance limitation. Categorical provider values use the same conservative agreement rule. Credential strings are not split or merged without a verified vocabulary.

Current patient attributes and diagnosis are supplied through an encounter-to-patient context table. This avoids overwriting a patient's age/insurance with future values. It does not introduce a direct Patient–Encounter graph edge. Current encounter clinical availability remains our approved assumption.


In [ ]:
if "selected" not in globals() or "train_candidates" not in globals():
    raise RuntimeError("Run the source-selection and candidate sections first.")
pcp_feature_sources={"practice_size":"practice size_pcp","years_in_medicare":"years in medicare_pcp",
                     "beneficiary_count":"Tot_Benes_pcp","credentials":"pcp_credentials_selected"}
specialist_feature_sources={"practice_size":"practice size","years_in_medicare":"years in medicare",
 "beneficiary_count":"Tot_Benes","service_count":"Tot_Srvcs","mips_score":"final MIPS score",
 "rating":"Rating_RP","rating_count":"Number_of_ratings_RP","average_panel_age":"Bene_Avg_Age",
 "average_panel_risk":"Bene_Avg_Risk_Scre","panel_age_lt65":"Bene_Age_LT_65_Cnt",
 "panel_age_65_74":"Bene_Age_65_74_Cnt","panel_age_75_84":"Bene_Age_75_84_Cnt",
 "panel_age_gt84":"Bene_Age_GT_84_Cnt","panel_female":"Bene_Feml_Cnt","panel_male":"Bene_Male_Cnt",
 "specialty":"specialist_specialty_selected","credentials":"specialist_credentials_selected"}
categorical_features={"credentials","specialty"}
provider_histories={}
provider_history_report=[]
for role,provider,sources in [("pcp","primary_care_provider_npi",pcp_feature_sources),
                             ("specialist","rendering_provider_npi",specialist_feature_sources)]:
    base=selected[[provider,"_service_day"]].drop_duplicates().rename(columns={provider:"provider_id"})
    base=base.sort_values(["provider_id","_service_day"]).reset_index(drop=True)
    for feature,source in sources.items():
        if source not in selected:
            raise ValueError("Missing selected source column: "+source)
        raw=selected[source]
        if feature in categorical_features:
            values=raw.astype("string").str.strip().str.casefold().mask(raw.astype("string").str.strip().eq(""))
            # Factor codes are internal equality keys, never predictive category encodings.
            codes,vocabulary=pd.factorize(values,sort=True)
            numeric=pd.Series(codes,index=selected.index,dtype="float64").mask(codes<0)
        else:
            numeric=pd.to_numeric(raw,errors="coerce")
            numeric=numeric.where(np.isfinite(numeric.fillna(0).astype(float)) & numeric.ge(0))
        work=pd.DataFrame({"provider_id":selected[provider],"_service_day":selected._service_day,"value":numeric})
        daily=work.groupby(["provider_id","_service_day"])["value"].agg(["min","max"]).reset_index()
        daily=daily.sort_values(["provider_id","_service_day"])
        lo=daily.groupby("provider_id")["min"].cummin().groupby(daily.provider_id).ffill()
        hi=daily.groupby("provider_id")["max"].cummax().groupby(daily.provider_id).ffill()
        ambiguous=lo.notna() & hi.notna() & lo.ne(hi)
        resolved=lo.where(~ambiguous)
        if feature in categorical_features:
            resolved=resolved.map(dict(enumerate(vocabulary))).astype("string")
        daily[feature]=resolved
        daily[feature+"_ambiguous"]=ambiguous
        base=base.merge(daily[["provider_id","_service_day",feature,feature+"_ambiguous"]],
                        on=["provider_id","_service_day"],how="left",validate="one_to_one")
        provider_history_report.append({"role":role,"feature":feature,
             "provider_days":len(daily),"ambiguous_provider_days":int(ambiguous.sum())})
    provider_histories[role]=base
display(pd.DataFrame(provider_history_report))

def provider_features_asof(queries,role):
    id_col={"pcp":"primary_care_provider_npi","specialist":"rendering_provider_npi"}[role]
    sources=pcp_feature_sources if role=="pcp" else specialist_feature_sources
    q=queries[[id_col,"_service_day"]].rename(columns={id_col:"provider_id"}).reset_index(drop=True)
    if q.isna().any().any(): raise ValueError("Provider feature query keys must be complete")
    q["_order"]=np.arange(len(q))
    h=provider_histories[role].rename(columns={"_service_day":"_observed_day"}).copy()
    dtype=pd.StringDtype(storage="python")
    q["provider_id"]=q.provider_id.astype(dtype); h["provider_id"]=h.provider_id.astype(dtype)
    q["_service_day"]=pd.to_datetime(q._service_day).astype("datetime64[ns]")
    h["_observed_day"]=pd.to_datetime(h._observed_day).astype("datetime64[ns]")
    merged=pd.merge_asof(q.sort_values("_service_day"),h.sort_values("_observed_day"),
        left_on="_service_day",right_on="_observed_day",by="provider_id",direction="backward",allow_exact_matches=False)
    merged=merged.sort_values("_order").reset_index(drop=True)
    out=pd.DataFrame(index=merged.index)
    for feature in sources:
        out[role+"_"+feature]=merged[feature]
        out[role+"_"+feature+"_missing"]=merged[feature].isna()
        out[role+"_"+feature+"_ambiguous"]=merged[feature+"_ambiguous"].eq(True)
    return out


In [ ]:
# Resolve current patient context only within the reconstructed encounter.
def unique_or_missing(series):
    values=series.dropna().unique()
    return values[0] if len(values)==1 else pd.NA

patient_nodes=events[["mso_membership_id"]].drop_duplicates().copy()
encounter_features=events[["encounter_id","_service_day","mso_membership_id","primary_care_provider_npi"]].copy()
encounter_features=encounter_features.merge(split_assignments,on="encounter_id",validate="one_to_one")
diagnoses=event_diagnosis_codes.rename("diagnosis_codes").reset_index()
encounter_features=encounter_features.merge(diagnoses,on="encounter_id",validate="one_to_one")
encounter_features["diagnosis_missing"]=encounter_features.diagnosis_codes.map(len).eq(0)
patient_context=encounter_features[["encounter_id","mso_membership_id"]].copy()
context_report=[]
for feature,source in [("age","Age"),("gender","Gender"),("insurance_company","IPA_CARRIER"),("aco_status","aco_status")]:
    v=selected[source].copy()
    if feature=="age":
        v=pd.to_numeric(v,errors="coerce")
        v=v.where(np.isfinite(v.fillna(0).astype(float)) & v.ge(0) & v.mod(1).eq(0))
    work=pd.DataFrame({"encounter_id":selected.encounter_id,"value":v})
    group=work.groupby("encounter_id").value
    resolved=group.agg(unique_or_missing)
    ambiguous=group.nunique().gt(1)
    patient_context[feature]=patient_context.encounter_id.map(resolved)
    patient_context[feature+"_missing"]=patient_context[feature].isna()
    patient_context[feature+"_ambiguous"]=patient_context.encounter_id.map(ambiguous).fillna(False)
    context_report.append({"feature":feature,"missing_events":int(patient_context[feature+"_missing"].sum()),
                          "ambiguous_events":int(patient_context[feature+"_ambiguous"].sum())})
display(pd.DataFrame(context_report))
referral_targets=events[["encounter_id","rendering_provider_npi"]].copy()
# Targets remain separate; never concatenate this table into encounter model inputs.
pcp_query_features=pd.concat([encounter_features[["encounter_id"]].reset_index(drop=True),
                              provider_features_asof(encounter_features,"pcp")],axis=1)

def attach_specialist_features(candidate_batch):
    batch=candidate_batch.reset_index(drop=True).copy()
    features=provider_features_asof(batch,"specialist")
    return pd.concat([batch,features],axis=1)

def iter_training_feature_batches(batch_size=10000):
    if batch_size<1: raise ValueError("batch_size must be positive")
    for start in range(0,len(train_candidates),batch_size):
        yield attach_specialist_features(train_candidates.iloc[start:start+batch_size])

def iter_evaluation_feature_batches(split_name,batch_size=10000):
    for batch in iter_evaluation_candidates_with_distance(split_name,batch_size):
        yield attach_specialist_features(batch)

# Bounded smoke check, not full evaluation or a dump of private feature tables.
probe=next(iter_training_feature_batches(),None)
print({"patient_nodes":len(patient_nodes),"encounter_feature_rows":len(encounter_features),
       "patient_context_rows":len(patient_context),"pcp_query_feature_rows":len(pcp_query_features),
       "training_feature_probe_rows":len(probe) if probe is not None else 0,
       "target_specialist_excluded_from_encounter_inputs":"rendering_provider_npi" not in encounter_features})
if probe is not None:
    display(pd.DataFrame([{"feature":f,"probe_missing_rows":int(probe["specialist_"+f+"_missing"].sum()),
                          "probe_ambiguous_rows":int(probe["specialist_"+f+"_ambiguous"].sum())}
                         for f in ["mips_score","practice_size"]]))
del probe
assert len(encounter_features)==len(events)==len(patient_context)==len(pcp_query_features)
print("Feature tables remain private in memory. Encoding, scaling, graph tensors and training are not yet performed.")


Save the aggregate outputs for review. Training/evaluation batches include candidate-level history, geographic PCP distance, specialist values and masks; patient_context and pcp_query_features join by encounter_id. The batch label and IDs must be explicitly excluded from predictive numeric features. Patient/encounter history aggregation is still an architecture task. Persistent disagreement remains missing even across later years under this baseline; revisiting that conservative policy requires source/version context, not arbitrary first/max/mean selection.


## 15. Final validation and versioned local save
Run earlier preparation sections first. Validation does not train, fit encoders, or compute the full validation/test candidate cross-product. Provider-feature missingness below is over unique candidate-provider/query-date combinations (not weighted by every candidate row). Patient and PCP coverage is measured over encounters. Training candidates are checked in full; evaluation assembly gets a bounded smoke check.

Private output: local pickle tables preserve nullable types and diagnosis tuples without extra packages. Only load pickle files you created and trust. No tables are uploaded. A new timestamped folder is created; existing bundles are never overwritten. Unmatched geography and explicitly masked ambiguities are accepted missing values, not blockers. Reported range anomalies are not silently clipped.


In [ ]:
import hashlib, json, platform
from datetime import datetime, timezone
required_objects=["encounter_features","patient_context","pcp_query_features","provider_histories",
 "train_candidates","referral_targets","split_assignments","row_event_map","pair_daily",
 "provider_location_maps","candidate_audit","event_candidates_source"]
if any(x not in globals() for x in required_objects):
    raise RuntimeError("Run all prior preparation sections before closeout.")
checks=[]
def check(name,condition):
    checks.append({"check":name,"passed":bool(condition)})
ids=set(encounter_features.encounter_id)
for name,table in [("encounters",encounter_features),("patient_context",patient_context),
                   ("pcp_features",pcp_query_features),("targets",referral_targets),("splits",split_assignments)]:
    check(name+" unique event keys",table.encounter_id.is_unique)
    check(name+" aligned event keys",set(table.encounter_id)==ids)
check("source-row mapping unique",row_event_map._source_row.is_unique)
check("source-row mapping valid events",set(row_event_map.encounter_id)==ids)
check("target excluded from encounter inputs","rendering_provider_npi" not in encounter_features)
check("training query membership",set(train_candidates.encounter_id).issubset(set(split_assignments.loc[split_assignments.split.eq("train"),"encounter_id"])))
check("candidate pairs unique",not train_candidates.duplicated(["encounter_id","rendering_provider_npi"]).any())
check("binary training labels",train_candidates.label.isin([0,1]).all())
check("one target per training query",train_candidates.groupby("encounter_id").label.sum().eq(1).all())
actual=referral_targets.set_index("encounter_id").rendering_provider_npi
check("training labels match held-out target",train_candidates.label.eq(train_candidates.rendering_provider_npi.eq(train_candidates.encounter_id.map(actual)).astype(int)).all())
check("candidate available strictly earlier",train_candidates.rendering_provider_npi.map(specialist_first_seen).lt(train_candidates._service_day).all())
check("one split per date",encounter_features.groupby("_service_day").split.nunique().le(1).all())
check("chronological boundaries",((encounter_features.split.eq("train") & encounter_features._service_day.dt.year.le(2017)) |
 (encounter_features.split.eq("validation") & encounter_features._service_day.dt.year.eq(2018)) |
 (encounter_features.split.eq("test") & encounter_features._service_day.dt.year.eq(2019))).all())
check("nonnegative integer prior counts",(train_candidates.prior_referral_count.ge(0) & train_candidates.prior_referral_count.mod(1).eq(0)).all())
has_history=train_candidates.prior_referral_count.gt(0)
check("recency matches history availability",train_candidates.days_since_last_referral.notna().eq(has_history).all())
check("recency strictly positive",train_candidates.days_since_last_referral.dropna().gt(0).all())
check("first history date strictly earlier",train_candidates.loc[has_history,"first_observed_referral_date"].lt(train_candidates.loc[has_history,"_service_day"]).all())
check("diagnosis tuple strings",encounter_features.diagnosis_codes.map(lambda x:isinstance(x,tuple) and all(isinstance(v,str) for v in x)).all())
distance=train_candidates.pcp_specialist_geo_distance_miles
check("geographic distances finite nonnegative or missing",(distance.isna() | (np.isfinite(distance) & distance.ge(0))).all())
check("distance missing mask correct",train_candidates.pcp_specialist_geo_distance_missing.eq(distance.isna()).all())
for split in ["validation","test"]:
    probe=next(iter_evaluation_feature_batches(split,1000),None)
    check(split+" feature batch constructed",probe is not None)
    if probe is not None: check(split+" distance field present","pcp_specialist_geo_distance_miles" in probe)
display(pd.DataFrame(checks))
if not all(x["passed"] for x in checks):
    raise ValueError("Validation failed: review the named checks before saving.")


In [ ]:
# Range diagnostics use source observations, so masking does not conceal anomalies.
range_reports=[]
range_specs={"Age":(0,120,True),"final MIPS score":(0,100,False),
             "Rating_RP":(0,5,False),"Number_of_ratings_RP":(0,None,True),
             "practice size":(0,None,True),"practice size_pcp":(0,None,True),
             "years in medicare":(0,None,True),"years in medicare_pcp":(0,None,True),
             "Tot_Benes":(0,None,True),"Tot_Benes_pcp":(0,None,True),"Tot_Srvcs":(0,None,False)}
for col,(low,high,integer) in range_specs.items():
    if col not in selected: continue
    v=pd.to_numeric(selected[col],errors="coerce")
    invalid=selected[col].notna() & (v.isna() | ~np.isfinite(v.fillna(0).astype(float)))
    out=v.lt(low) | (v.gt(high) if high is not None else False)
    fractional=v.notna() & v.mod(1).ne(0) if integer else pd.Series(False,index=v.index)
    range_reports.append({"column":col,"parse_or_finite_issues":int(invalid.sum()),
                          "outside_provisional_range":int(out.sum()),"unexpected_fractional_rows":int(fractional.sum())})
display(pd.DataFrame(range_reports))
print("Age 0–120 and rating 0–5 are provisional review ranges, not proof of source coding. No clipping applied.")

coverage_reports=[]
for split in ["train","validation","test"]:
    split_ids=set(split_assignments.loc[split_assignments.split.eq(split),"encounter_id"])
    for table_name,table,features in [("patient_context",patient_context,["age","gender","insurance_company","aco_status"]),
                                    ("pcp_query_features",pcp_query_features,["pcp_"+f for f in pcp_feature_sources])]:
        sub=table.loc[table.encounter_id.isin(split_ids)]
        for f in features:
            coverage_reports.append({"split":split,"table":table_name,"feature":f,"unit":"encounter",
              "rows":len(sub),"missing":int(sub[f].isna().sum()),
              "missing_pct":round(100*sub[f].isna().mean(),2) if len(sub) else None})
    dates=event_candidates_source.loc[event_candidates_source.split.eq(split),"_service_day"].drop_duplicates()
    counts={f:[0,0] for f in specialist_feature_sources}
    for date in dates:
        available=available_specialists(date)
        if len(available)==0: continue
        q=pd.DataFrame({"rendering_provider_npi":available,"_service_day":date})
        feat=provider_features_asof(q,"specialist")
        for f in counts:
            counts[f][0]+=len(feat); counts[f][1]+=int(feat["specialist_"+f].isna().sum())
    for f,(total,missing) in counts.items():
        coverage_reports.append({"split":split,"table":"specialist_asof","feature":f,"unit":"available specialist/date",
           "rows":total,"missing":missing,"missing_pct":round(100*missing/total,2) if total else None})
display(pd.DataFrame(coverage_reports))
print("Validation complete. Inspect range diagnostics before running the save cell.")


### Save the prepared local bundle
Run after reviewing validation. Range anomalies require an explicit local acknowledgement; they are recorded in the manifest, not hidden. Saving writes private data only to the folder below, with a completion marker written last. A partial folder without COMPLETE.json is not a finished bundle. Evaluation candidates remain lazy: their query rows, provider availability, pair history and feature histories are saved, rather than hundreds of millions of expanded rows.


In [ ]:
SAVE_PREPARED_BUNDLE=True  # User-authorized local save; set False to inspect only.
ACKNOWLEDGE_RANGE_FLAGS=False  # Change only after reviewing flags above; they remain recorded.
BUNDLE_ROOT=Path("prepared_data")
range_flags=sum(r["parse_or_finite_issues"]+r["outside_provisional_range"]+r["unexpected_fractional_rows"] for r in range_reports)
if SAVE_PREPARED_BUNDLE:
    if not all(c["passed"] for c in checks): raise ValueError("Integrity checks must pass before save")
    if range_flags and not ACKNOWLEDGE_RANGE_FLAGS:
        raise ValueError("Range diagnostics flagged values; review before acknowledging and saving.")
    stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    bundle=BUNDLE_ROOT/("referral_prepared_"+stamp)
    bundle.mkdir(parents=True,exist_ok=False)
    tables={"patient_nodes":patient_nodes,"encounter_features":encounter_features,"patient_context":patient_context,
      "pcp_query_features":pcp_query_features,"referral_targets":referral_targets,"split_assignments":split_assignments,
      "row_event_map":row_event_map,"pair_daily":pair_daily,"train_candidates":train_candidates,
      "candidate_audit":candidate_audit,"evaluation_queries":event_candidates_source,
      "pcp_history":provider_histories["pcp"],"specialist_history":provider_histories["specialist"],
      "pcp_locations":provider_location_maps["pcp"],"specialist_locations":provider_location_maps["specialist"],
      "specialist_first_seen":specialist_first_seen.to_frame("first_seen")}
    def digest_file(path):
        h=hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda:stream.read(1024*1024),b""): h.update(block)
        return h.hexdigest()
    files={}
    for name,table in tables.items():
        path=bundle/(name+"_PRIVATE.pkl")
        table.to_pickle(path)
        files[path.name]={"rows":len(table),"sha256":digest_file(path),
           "columns":{str(c):str(t) for c,t in table.dtypes.items()}}
    # Copy source code without saved outputs, executed counts, or metadata.
    notebook_path=Path("01_data_cleaning.ipynb")
    code_hash=None
    if notebook_path.is_file():
        nb=json.loads(notebook_path.read_text())
        code="\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"]=="code")
        (bundle/"preparation_code_PRIVATE.txt").write_text(code)
        code_hash=hashlib.sha256(code.encode()).hexdigest()
    manifest={"created_utc":stamp,"schema_version":"initial-baseline-v1",
      "source_filename":DATA_PATH.name,"source_sheet":SHEET_NAME,"source_sha256":digest_file(DATA_PATH),
      "python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,
      "code_sha256":code_hash,"tables":files,"checks":checks,"range_checks":range_reports,
      "coverage":coverage_reports,"split":{"train":"2014–2017","validation":"2018","test":"observed 2019"},
      "event_key":["mso_membership_id","primary_care_provider_npi","service_from_date (day)","rendering_provider_npi"],
      "history":"strictly earlier dates; rolling completed-event history; same-day excluded",
      "candidates":{"availability":"specialist first observed date strictly before query","seed":CANDIDATE_SEED,
                    "training_alternatives":NEGATIVES_PER_QUERY,"evaluation":"all available; unavailable target counts as miss"},
      "pcp_feature_sources":pcp_feature_sources,"specialist_feature_sources":specialist_feature_sources,
      "insurance_mapping":product_map,"missing_policy":"null plus masks; cumulative conflicting provider values masked",
      "geography":{"reference":REFERENCE_URL,"sha256":digest_file(REFERENCE_PATH),"units":"great-circle miles",
                    "pcp_zip":"Provider_Zip5Texas_pcp","specialist_zip":"Provider_Zip5Texas","snapshot_assumption":True},
      "limitations":["diagnosis assumed available before referral","service date is referral-time proxy",
        "provider snapshots not historically verified","same-day real visits may merge","patient distance unavailable",
        "ZIP/ZCTA representative points are not driving distance","conflict rule retains persistent ambiguity",
        "encoders, graph tensors and clinical-suitability evaluation not implemented"]}
    (bundle/"manifest_PRIVATE.json").write_text(json.dumps(manifest,indent=2))
    (bundle/"COMPLETE.json").write_text(json.dumps({"manifest_sha256":digest_file(bundle/"manifest_PRIVATE.json"),"tables":len(files)}))
    print({"bundle_saved":True,"bundle_folder":str(bundle),"tables":len(files)})
else:
    print("Save disabled; prepared tables remain in memory.")
